# Abstraction 

We implement a multi-stage pipeline to summarize a pickleball video by simulating **ball**, **event**, and **audio** detectors, aligning their outputs, and extracting highlights. This multimodal approach is inspired by sports highlight curation research, which fuses information from player actions, crowd cheers, and commentary to identify exciting moments. Audio cues alone (like crowd noise and ball hits) have also been shown to effectively detect tennis events. The pipeline steps are:

- Load Video: Use OpenCV or MoviePy to read the input MP4 and obtain its duration and frame rate.
- Simulate Detections: Generate timestamps (or frame indices) for three “detectors”: Ball Detection (e.g. YOLO-like; random or patterned hit frames), Event Detection (e.g. rallies/point ends), and Audio Detection (e.g. crowd cheers after points).
- Temporal Alignment: Combine all detected timestamps and cluster them by time. We consider a short time window (e.g. 1 sec) as “aligned” if signals from different modalities fall into it. This mimics fusing multimodal excitement features (players, spectators, audio) to find highlight-worthy moments ￼.
- Identify “Vibe” Moments: Define a vibe moment as any time where at least two detectors coincide within the time window (e.g. a ball hit and a cheer).
- Extract Highlights: For each vibe moment, extract a short subclip (e.g. 4–6 seconds around that time) from the original video using MoviePy. MoviePy’s VideoFileClip.subclip(start, end) makes this straightforward ￼. Save each highlight as an MP4 file.

<img src="pickleball_vibe.png" style="display: block; margin: 0 auto;"  />

# Implementation

In [ ]:
import cv2
import numpy as np
from moviepy.video.io.VideoFileClip import VideoFileClip

# -------------------------------
# 1. Load video and get properties
# -------------------------------
input_video = "../data/input_tennis_match.mp4"  # Replace with your video file path
cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise IOError(f"Cannot open video file {input_video}")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
duration = frame_count / fps
cap.release()
print(f"Loaded video: duration={duration:.2f}s, FPS={fps}")


# -------------------------------
# 2. Simulate detection outputs
# -------------------------------
# (a) Simulate ball detection timestamps (in seconds)
np.random.seed(0)
ball_times = list(np.random.uniform(0, duration, size=8))
# (b) Simulate event detection (e.g. end of rallies)
event_times = list(np.random.uniform(0, duration, size=5))
# (c) Simulate audio cues (crowd cheers, silence)
audio_times = list(np.random.uniform(0, duration, size=6))

# (Optional) Add structure: cluster around some points to create overlaps
base_times = np.linspace(10, duration-10, num=3)
for base in base_times:
    event_times.append(base)
    ball_times += [base - 0.5, base + 0.3]   # ball hits around the event
    audio_times.append(base + 1.0)           # cheer shortly after
# Sort all lists
ball_times = sorted([t for t in ball_times if 0 <= t <= duration])
event_times = sorted([t for t in event_times if 0 <= t <= duration])
audio_times = sorted([t for t in audio_times if 0 <= t <= duration])
print("Simulated Ball hits:", ball_times)
print("Simulated Events:", event_times)
print("Simulated Audio cues:", audio_times)

# -------------------------------
# 3. Temporal alignment and 'vibe' detection
# -------------------------------
# Combine all signals into one timeline list
signals = []
for t in ball_times:
    signals.append((t, 'ball'))
for t in event_times:
    signals.append((t, 'event'))
for t in audio_times:
    signals.append((t, 'audio'))
signals.sort(key=lambda x: x[0])

# Cluster signals by a time threshold and count modalities in each cluster
vibe_times = []
threshold = 1.0  # seconds window to consider as aligned
i = 0
while i < len(signals):
    cluster = [signals[i]]
    j = i + 1
    # Group all signals within `threshold` of each other
    while j < len(signals) and signals[j][0] - cluster[-1][0] <= threshold:
        cluster.append(signals[j])
        j += 1
    # Count how many unique modalities are in this cluster
    modalities = {mod for (_, mod) in cluster}
    if len(modalities) >= 2:
        # Define vibe time as the center of the cluster
        times = [t for (t, _) in cluster]
        vibe_time = sum(times) / len(times)
        vibe_times.append(vibe_time)
    i = j

vibe_times = sorted(vibe_times)
print("Detected vibe moments at (s):", vibe_times)

# -------------------------------
# 4. Extract highlight subclips
# -------------------------------
# Use MoviePy to cut short clips around each vibe time
clip = VideoFileClip(input_video)
highlight_duration = 4.0  # seconds before and after vibe time
for idx, t in enumerate(vibe_times):
    start = max(0, t - highlight_duration/2)
    end = min(duration, t + highlight_duration/2)
    subclip = clip.subclipped(start, end)
    out_filename = f"highlight_{idx+1}.mp4"
    print(f"Writing highlight {idx+1}: {start:.2f}-{end:.2f}s to {out_filename}")
    subclip.write_videofile(out_filename, codec="libx264")
clip.close()

Loaded video: duration=1566.47s, FPS=30.0
